# SCRIPT 2B — SEGFORMER TRAINING
## Brahmaputra Morphodynamics Study (AP-GARSS 2026)

**Purpose:** Train SegFormer-B2 for 4-class braidplain segmentation.
Compare against Script 2A Attention U-Net using:
- Test tile Dice/mIoU/BIoU (training reach, 2024)
- Morphological integrity assessment via cross-sections (inference reach, 2025)

**Architecture:** SegFormer-B2 (Mix Transformer encoder + lightweight MLP decoder)
- Pretrained encoder weights from ImageNet via HuggingFace
- Input projection modified for 9 channels
- Output head modified for 4 classes
- Global attention mechanism — handles spatial context beyond local receptive field

**Classes:** 0=Background  1=Active Channel  2=Active Bar  3=Vegetated Bar

**Model selection:** Best combined score on test tiles + morphological integrity
assessment determines which model proceeds to Script 3.

## 0. Install and Imports

In [ ]:
!pip install albumentations torchmetrics rasterio transformers -q

import os
import json
import gc
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import albumentations as A
from albumentations.pytorch import ToTensorV2
import rasterio
from scipy.ndimage import binary_erosion, binary_dilation
from transformers import SegformerForSemanticSegmentation, SegformerConfig

from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Imports complete.')

## 1. Configuration

In [ ]:
# ── Paths — identical structure to Script 2A, separate output folder ──────
ROOT          = '/content/drive/MyDrive/BRAHMA_MORPHO'
TILES_DIR     = f'{ROOT}/training/tiles/training'
TILES_DIR2    = f'{ROOT}/training/tiles/training_2025'
SPLITS_DIR    = f'{ROOT}/training/splits'
OUTPUTS_DIR   = f'{ROOT}/models/segformer'
FIGURES_DIR   = f'{ROOT}/figures/segformer'
INFERENCE_DIR = f'{ROOT}/inference/scenes'
# Reuse norm stats from Script 2A — same training tiles
ATTN_OUTPUTS  = f'{ROOT}/models/attention_unet'

for d in [OUTPUTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────
NUM_CLASSES         = 4
IN_CHANNELS         = 9
TILE_SIZE           = 256
BATCH_SIZE          = 4      # SegFormer uses more VRAM than Attention U-Net
                             # reduce to 2 if OOM on T4
NUM_EPOCHS          = 100
LEARNING_RATE       = 6e-5   # lower LR for pretrained transformer encoder
ENCODER_LR_FACTOR   = 0.1    # encoder LR = LEARNING_RATE * ENCODER_LR_FACTOR
                             # fine-tune encoder slowly, decoder faster
WEIGHT_DECAY        = 1e-4
EARLY_STOP_PATIENCE = 15
LR_PATIENCE         = 5
EDGE_WEIGHT         = 5.0
SEED                = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

BAND_NAMES   = ['B2','B3','B4','B8','B8A','B11','B12','VV','VH']
CLASS_NAMES  = ['Background','Active Channel','Active Bar','Vegetated Bar']
CLASS_COLORS = ['#1A0400','#66CCFF','#E6F2FF','#66FFB3']

print('Configuration set.')
print(f'Batch size  : {BATCH_SIZE}  (reduce to 2 if GPU OOM)')
print(f'LR encoder  : {LEARNING_RATE * ENCODER_LR_FACTOR:.2e}')
print(f'LR decoder  : {LEARNING_RATE:.2e}')

## 2. Normalisation Statistics (Reuse from Script 2A)

In [ ]:
# ── Load pre-computed norm stats from Script 2A ───────────────────────────
# Same training tiles — no need to recompute
NORM_MEAN = np.load(os.path.join(ATTN_OUTPUTS, 'norm_mean.npy'))
NORM_STD  = np.load(os.path.join(ATTN_OUTPUTS, 'norm_std.npy'))

print('Normalisation stats loaded from Script 2A:')
print(f'{"Band":6s}  {"Mean":>10s}  {"Std":>10s}')
print('-'*30)
for i, name in enumerate(BAND_NAMES):
    print(f'{name:6s}  {NORM_MEAN[i]:>10.2f}  {NORM_STD[i]:>10.2f}')

## 3. Dataset and DataLoader

In [ ]:
# ── Augmentation — identical to Script 2A ────────────────────────────────
def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.0, scale_limit=0.15,
            rotate_limit=0,  border_mode=0, p=0.3
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.15, contrast_limit=0.15, p=0.4
        ),
        A.GaussNoise(p=0.2),
        ToTensorV2()
    ], additional_targets={'mask': 'mask'})

def get_val_transforms():
    return A.Compose([ToTensorV2()],
                     additional_targets={'mask': 'mask'})




In [ ]:
def get_tile_dir(tile_id):
    """Route tile_id to correct directory based on prefix."""
    if tile_id.startswith('train2025'):
        return os.path.join(TILES_DIR2, tile_id)
    else:
        return os.path.join(TILES_DIR, tile_id)


# ── Dataset class ─────────────────────────────────────────────────────────
class BraidplainDataset(Dataset):
    def __init__(self, tile_ids, mean, std, transform=None):
        self.mean      = mean
        self.std       = std
        self.transform = transform

        # Filter to tiles that exist on disk across both directories
        self.tile_ids = [
            t for t in tile_ids
            if os.path.exists(os.path.join(get_tile_dir(t), 'image.npy'))
        ]
        print(f'  Dataset: {len(self.tile_ids)} tiles loaded '
              f'({len(tile_ids) - len(self.tile_ids)} missing on disk)')

    def __len__(self):
        return len(self.tile_ids)

    def __getitem__(self, idx):
        tile_id  = self.tile_ids[idx]
        tile_dir = get_tile_dir(tile_id)   # routes to correct folder

        image = np.load(os.path.join(tile_dir,'image.npy')).astype(np.float32)
        mask  = np.load(os.path.join(tile_dir,'mask.npy'))[:,:,0].astype(np.int64)

        for c in range(IN_CHANNELS):
            nodata = (image[:,:,c] == 0)
            if nodata.any():
                image[:,:,c][nodata] = self.mean[c]
            image[:,:,c] = (image[:,:,c] - self.mean[c]) / self.std[c]

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image     = augmented['image']
            mask      = augmented['mask']
        else:
            image = torch.from_numpy(image.transpose(2,0,1))
            mask  = torch.from_numpy(mask)

        return image.float(), mask.long()


# ── Load splits and create datasets ──────────────────────────────────────
with open(os.path.join(SPLITS_DIR,'train_tiles.json')) as f:
    train_ids = json.load(f)
with open(os.path.join(SPLITS_DIR,'val_tiles.json')) as f:
    val_ids   = json.load(f)
with open(os.path.join(SPLITS_DIR,'test_tiles.json')) as f:
    test_ids  = json.load(f)

print('Creating datasets...')
# Note: tiles_dir argument removed — routing handled by get_tile_dir()
train_dataset = BraidplainDataset(train_ids, NORM_MEAN, NORM_STD,
                                   transform=get_train_transforms())
val_dataset   = BraidplainDataset(val_ids,   NORM_MEAN, NORM_STD,
                                   transform=get_val_transforms())
test_dataset  = BraidplainDataset(test_ids,  NORM_MEAN, NORM_STD,
                                   transform=get_val_transforms())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'\nDataLoaders ready:')
print(f'  Train : {len(train_dataset)} tiles, {len(train_loader)} batches')
print(f'  Val   : {len(val_dataset)} tiles, {len(val_loader)} batches')
print(f'  Test  : {len(test_dataset)} tiles, {len(test_loader)} batches')

# Verify both years present in training
n_2025 = sum(1 for t in train_dataset.tile_ids if t.startswith('train2025'))
n_2024 = len(train_dataset.tile_ids) - n_2025
print(f'\n  Training breakdown: {n_2024} from 2024 + {n_2025} from 2025')

## 4. Class Weights

In [ ]:
def compute_class_weights(tiles_dir, tile_ids, num_classes=4):
    counts = np.zeros(num_classes, dtype=np.int64)
    for tile_id in tqdm(tile_ids, desc='Class weights'):
        mask_path = os.path.join(tiles_dir, tile_id, 'mask.npy')
        if not os.path.exists(mask_path):
            continue
        mask = np.load(mask_path)[:,:,0]
        for c in range(num_classes):
            counts[c] += int(np.sum(mask == c))
    total   = counts.sum()
    weights = total / (num_classes * counts)
    weights = weights / weights.mean()
    print('Class weights:')
    for c, (name, w, n) in enumerate(zip(CLASS_NAMES, weights, counts)):
        print(f'  {c} {name:16s}: {w:.3f}  ({100*n/total:.1f}%)')
    return torch.tensor(weights, dtype=torch.float32)

CLASS_WEIGHTS = compute_class_weights(TILES_DIR, train_ids).to(device)

## 5. SegFormer-B2 Model

In [ ]:
# ── SegFormer-B2 with 9-channel input ────────────────────────────────────
# Architecture: Mix Transformer (MiT-B2) encoder + lightweight MLP decoder
# Key advantage over Attention U-Net:
#   - Hierarchical transformer encoder with overlapping patch embeddings
#   - Mix-FFN uses local convolutions inside transformer blocks
#   - Global self-attention at coarser scales — captures long-range context
#   - This long-range context is the key test for vegetated bar disambiguation
#
# Input modification: original SegFormer takes 3 channels (RGB)
# We replace the patch embedding projection to accept 9 channels
# Pretrained weights initialised for 3-channel input — we handle
# the mismatch by averaging the 3-channel projection and replicating
# then fine-tuning from there

def build_segformer(in_channels=9, num_classes=4):
    """
    Build SegFormer-B2 with modified input for 9-channel remote sensing data.

    Strategy for 9-channel input:
    1. Load pretrained SegFormer-B2 config
    2. Modify num_channels to 9
    3. Initialise from pretrained weights where possible
    4. Re-initialise the first patch embedding projection for 9 channels
       using the mean of the original 3-channel weights repeated 3 times
       This preserves the learned spatial feature patterns as initialisation
    """
    print('Building SegFormer-B2 with 9-channel input...')

    # ── Load pretrained config ────────────────────────────────────────
    config = SegformerConfig.from_pretrained(
        'nvidia/mit-b2',
        num_labels       = num_classes,
        num_channels     = in_channels,
        ignore_mismatches= True
    )

    # ── Initialise model with modified config ─────────────────────────
    # This creates the model architecture with 9-channel input
    # but random weights (pretrained weights incompatible with 9ch input)
    model = SegformerForSemanticSegmentation(config)

    # ── Load pretrained encoder weights for all layers except patch embed ─
    # Load pretrained 3-channel model
    try:
        pretrained = SegformerForSemanticSegmentation.from_pretrained(
            'nvidia/mit-b2',
            num_labels   = num_classes,
            ignore_mismatches = True
        )
        print('  Pretrained MiT-B2 weights loaded.')

        # Copy all weights except first patch embedding projection
        pretrained_state = pretrained.state_dict()
        model_state      = model.state_dict()

        n_copied  = 0
        n_skipped = 0

        for key in model_state:
            if key in pretrained_state:
                if pretrained_state[key].shape == model_state[key].shape:
                    model_state[key] = pretrained_state[key]
                    n_copied += 1
                else:
                    # Shape mismatch — this is the first patch embedding
                    # Re-initialise for 9 channels using pretrained 3ch weights
                    if 'patch_embeddings.0.proj.weight' in key:
                        # Original: (C_out, 3, kH, kW)
                        # Target:   (C_out, 9, kH, kW)
                        # Strategy: tile the 3-channel weights 3 times and scale
                        w3  = pretrained_state[key]  # (C_out, 3, kH, kW)
                        w9  = w3.repeat(1, 3, 1, 1) / 3.0  # (C_out, 9, kH, kW)
                        model_state[key] = w9
                        print(f'  Patch embed projection: 3ch→9ch via tiling')
                    n_skipped += 1
            else:
                n_skipped += 1

        model.load_state_dict(model_state)
        print(f'  Weights copied: {n_copied}  Skipped/re-init: {n_skipped}')
        del pretrained, pretrained_state
        gc.collect()

    except Exception as e:
        print(f'  WARNING: Could not load pretrained weights: {e}')
        print('  Training from random initialisation.')

    return model


model = build_segformer(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nSegFormer-B2 ready.')
print(f'Total parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# ── Quick forward pass test ───────────────────────────────────────────────
dummy = torch.randn(2, IN_CHANNELS, TILE_SIZE, TILE_SIZE).to(device)
with torch.no_grad():
    out = model(pixel_values=dummy)
    # SegFormer returns logits at 1/4 resolution — must upsample
    logits = F.interpolate(
        out.logits,
        size   = (TILE_SIZE, TILE_SIZE),
        mode   = 'bilinear',
        align_corners = False
    )
print(f'Forward pass: input {list(dummy.shape)} → logits {list(logits.shape)}')
del dummy, out, logits
torch.cuda.empty_cache()

## 6. Loss Function (Identical to Script 2A)

In [ ]:
class EdgeWeightedCELoss(nn.Module):
    def __init__(self, class_weights, edge_weight=5.0):
        super().__init__()
        self.ce          = nn.CrossEntropyLoss(weight=class_weights, reduction='none')
        self.edge_weight = edge_weight

    def forward(self, logits, targets):
        loss       = self.ce(logits, targets)
        edge_map   = torch.zeros_like(targets, dtype=torch.float32)
        targets_np = targets.cpu().numpy()
        for b in range(targets.shape[0]):
            for c in range(NUM_CLASSES):
                cm = (targets_np[b] == c).astype(np.uint8)
                if cm.sum() == 0:
                    continue
                eroded   = binary_erosion(cm, iterations=1)
                boundary = cm - eroded.astype(np.uint8)
                edge_map[b] += torch.from_numpy(boundary).float().to(targets.device)
        weight_map    = 1.0 + (self.edge_weight - 1.0) * (edge_map > 0).float()
        return (loss * weight_map).mean()


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs      = F.softmax(logits, dim=1)
        targets_oh = F.one_hot(targets, NUM_CLASSES).permute(0,3,1,2).float()
        dice_losses = []
        for c in range(NUM_CLASSES):
            p  = probs[:,c].reshape(-1)
            t  = targets_oh[:,c].reshape(-1)
            intersection = (p * t).sum()
            dice = (2*intersection + self.smooth) / (p.sum() + t.sum() + self.smooth)
            dice_losses.append(1 - dice)
        return torch.stack(dice_losses).mean()


class CombinedLoss(nn.Module):
    def __init__(self, class_weights, edge_weight=5.0, alpha=0.6):
        super().__init__()
        self.ce_loss   = EdgeWeightedCELoss(class_weights, edge_weight)
        self.dice_loss = DiceLoss()
        self.alpha     = alpha

    def forward(self, logits, targets):
        return self.alpha * self.ce_loss(logits, targets) + \
               (1 - self.alpha) * self.dice_loss(logits, targets)


criterion = CombinedLoss(
    class_weights = CLASS_WEIGHTS,
    edge_weight   = EDGE_WEIGHT,
    alpha         = 0.6
).to(device)
print('Combined loss defined (EdgeWeightedCE + Dice, alpha=0.6)')

## 7. Evaluation Metrics (Identical to Script 2A)

In [ ]:
def dice_coefficient(pred, target, num_classes=NUM_CLASSES, smooth=1e-6):
    dice_pc = []
    for c in range(num_classes):
        p = (pred==c).astype(np.float32)
        t = (target==c).astype(np.float32)
        intersection = (p*t).sum()
        dice_pc.append(float((2*intersection+smooth)/(p.sum()+t.sum()+smooth)))
    return dice_pc, float(np.mean(dice_pc))

def mean_iou(pred, target, num_classes=NUM_CLASSES, smooth=1e-6):
    iou_pc = []
    for c in range(num_classes):
        p  = (pred==c).astype(np.float32)
        t  = (target==c).astype(np.float32)
        intersection = (p*t).sum()
        union = p.sum()+t.sum()-intersection
        iou_pc.append(float((intersection+smooth)/(union+smooth)))
    return iou_pc, float(np.mean(iou_pc))

def boundary_iou_class(pred_c, target_c, smooth=1e-6):
    def get_boundary(mask):
        if mask.sum()==0: return mask.astype(np.float32)
        eroded   = binary_erosion(mask, iterations=2)
        boundary = mask.astype(np.float32) - eroded.astype(np.float32)
        return binary_dilation(boundary>0, iterations=2).astype(np.float32)
    b_pred   = get_boundary(pred_c.astype(bool))
    b_target = get_boundary(target_c.astype(bool))
    intersection = (b_pred*b_target).sum()
    union = b_pred.sum()+b_target.sum()-intersection
    return float((intersection+smooth)/(union+smooth))

def boundary_iou_all_classes(pred, target, num_classes=NUM_CLASSES):
    biou_pc = [boundary_iou_class((pred==c),(target==c)) for c in range(num_classes)]
    return biou_pc, float(np.mean(biou_pc))

def batch_dice(logits, targets, smooth=1e-6):
    preds      = logits.argmax(dim=1)
    targets_oh = F.one_hot(targets,    NUM_CLASSES).permute(0,3,1,2).float()
    preds_oh   = F.one_hot(preds,      NUM_CLASSES).permute(0,3,1,2).float()
    intersection = (preds_oh * targets_oh).sum(dim=(0,2,3))
    union        = preds_oh.sum(dim=(0,2,3)) + targets_oh.sum(dim=(0,2,3))
    dice         = (2*intersection+smooth)/(union+smooth)
    return dice.mean().item()

print('Metrics defined: Dice, mIoU, BIoU, batch_dice')

## 8. Optimiser — Differential Learning Rates

In [ ]:
# ── Differential learning rates ───────────────────────────────────────────
# SegFormer has a pretrained encoder — fine-tune slowly
# The decode head is randomly initialised — train faster
# This is standard practice for transformer fine-tuning

encoder_params = []
decoder_params = []

for name, param in model.named_parameters():
    if 'segformer.encoder' in name:
        encoder_params.append(param)
    else:
        decoder_params.append(param)

print(f'Encoder parameters: {sum(p.numel() for p in encoder_params):,}')
print(f'Decoder parameters: {sum(p.numel() for p in decoder_params):,}')

optimiser = torch.optim.AdamW([
    {'params': encoder_params,
     'lr'    : LEARNING_RATE * ENCODER_LR_FACTOR,
     'name'  : 'encoder'},
    {'params': decoder_params,
     'lr'    : LEARNING_RATE,
     'name'  : 'decoder'}
], weight_decay=WEIGHT_DECAY)

scheduler = ReduceLROnPlateau(
    optimiser, mode='min', factor=0.5,
    patience=LR_PATIENCE,
    min_lr    = 1e-7
)

print(f'\nOptimiser: AdamW with differential LR')
print(f'  Encoder LR : {LEARNING_RATE * ENCODER_LR_FACTOR:.2e}')
print(f'  Decoder LR : {LEARNING_RATE:.2e}')
print(f'Scheduler  : ReduceLROnPlateau (factor=0.5, patience={LR_PATIENCE})')

## 9. Training Loop

In [ ]:
def segformer_forward(model, images):
    """
    SegFormer forward pass with upsampling.
    SegFormer outputs logits at 1/4 input resolution.
    Must upsample to full tile size before loss computation.
    """
    out    = model(pixel_values=images)
    logits = F.interpolate(
        out.logits,
        size          = (TILE_SIZE, TILE_SIZE),
        mode          = 'bilinear',
        align_corners = False
    )
    return logits


def train_epoch(model, loader, criterion, optimiser, device):
    model.train()
    total_loss = 0.0
    total_dice = 0.0
    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device,  non_blocking=True)
        optimiser.zero_grad()
        logits = segformer_forward(model, images)
        loss   = criterion(logits, masks)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        total_loss += loss.item()
        with torch.no_grad():
            total_dice += batch_dice(logits, masks)
    n = len(loader)
    return total_loss/n, total_dice/n


def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_dice = 0.0
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks  = masks.to(device,  non_blocking=True)
            logits = segformer_forward(model, images)
            loss   = criterion(logits, masks)
            total_loss += loss.item()
            total_dice += batch_dice(logits, masks)
    n = len(loader)
    return total_loss/n, total_dice/n


print('Training functions defined.')
print('NOTE: SegFormer logits are upsampled to 256x256 before loss computation.')

In [ ]:
# ── Main training loop ────────────────────────────────────────────────────
history = {
    'train_loss':[], 'val_loss':[],
    'train_dice':[], 'val_dice':[],
    'lr_encoder':[], 'lr_decoder':[]
}

best_val_dice    = 0.0
early_stop_count = 0
best_epoch       = 0
CHECKPOINT_PATH  = os.path.join(OUTPUTS_DIR, 'segformer_best.pth')

print('='*65)
print('TRAINING SEGFORMER-B2')
print(f'Max epochs     : {NUM_EPOCHS}')
print(f'Early stopping : patience={EARLY_STOP_PATIENCE}')
print(f'Checkpoint     : {CHECKPOINT_PATH}')
print('='*65)

for epoch in range(1, NUM_EPOCHS + 1):

    train_loss, train_dice = train_epoch(model, train_loader,
                                          criterion, optimiser, device)
    val_loss,   val_dice   = validate_epoch(model, val_loader,
                                             criterion, device)

    scheduler.step(val_loss)
    lr_enc = optimiser.param_groups[0]['lr']
    lr_dec = optimiser.param_groups[1]['lr']

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_dice'].append(train_dice)
    history['val_dice'].append(val_dice)
    history['lr_encoder'].append(lr_enc)
    history['lr_decoder'].append(lr_dec)

    print(f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
          f'Train loss: {train_loss:.4f} dice: {train_dice:.4f} | '
          f'Val loss: {val_loss:.4f} dice: {val_dice:.4f} | '
          f'LR enc: {lr_enc:.1e} dec: {lr_dec:.1e}')

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_epoch    = epoch
        torch.save({
            'epoch'       : epoch,
            'model_state' : model.state_dict(),
            'optim_state' : optimiser.state_dict(),
            'val_dice'    : val_dice,
            'val_loss'    : val_loss,
            'norm_mean'   : NORM_MEAN,
            'norm_std'    : NORM_STD,
            'num_classes' : NUM_CLASSES,
            'in_channels' : IN_CHANNELS,
        }, CHECKPOINT_PATH)
        print(f'  ✓ Best checkpoint saved (val_dice={val_dice:.4f})')
        early_stop_count = 0
    else:
        early_stop_count += 1

    if early_stop_count >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

print(f'\nTraining complete. Best epoch: {best_epoch}  Val Dice: {best_val_dice:.4f}')

with open(os.path.join(OUTPUTS_DIR,'segformer_history.json'),'w') as f:
    json.dump(history, f, indent=2)
print('History saved.')

## 10. Training Curves

In [ ]:
def plot_training_curves(history, save_path):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('SegFormer-B2 — Training Curves', fontsize=13, fontweight='bold')
    epochs = range(1, len(history['train_loss'])+1)

    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=1.5)
    axes[0].plot(epochs, history['val_loss'],   'r-', label='Val',   linewidth=1.5)
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['train_dice'], 'b-', label='Train', linewidth=1.5)
    axes[1].plot(epochs, history['val_dice'],   'r-', label='Val',   linewidth=1.5)
    axes[1].axhline(y=max(history['val_dice']), color='g', linestyle='--',
                    alpha=0.7, label=f'Best={max(history["val_dice"]):.4f}')
    axes[1].set_title('Mean Dice')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    axes[2].semilogy(epochs, history['lr_decoder'], 'g-',
                     label='Decoder LR', linewidth=1.5)
    axes[2].semilogy(epochs, history['lr_encoder'], 'b--',
                     label='Encoder LR', linewidth=1.5)
    axes[2].set_title('Learning Rate')
    axes[2].set_xlabel('Epoch')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved → {save_path}')

plot_training_curves(
    history,
    os.path.join(FIGURES_DIR, 'segformer_training_curves.png')
)

## 11. Test Tile Evaluation

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────
CHECKPOINT_PATH  = os.path.join(OUTPUTS_DIR, 'segformer_best.pth')
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f'Best checkpoint loaded (epoch {checkpoint["epoch"]}, '
      f'val_dice={checkpoint["val_dice"]:.4f})')


def evaluate_on_tiles(model, loader, device, split_name='Test'):
    all_preds, all_targets = [], []
    model.eval()
    with torch.no_grad():
        for images, masks in tqdm(loader, desc=f'Evaluating {split_name}'):
            images = images.to(device)
            logits = segformer_forward(model, images)
            preds  = logits.argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(masks.numpy())

    all_preds   = np.concatenate(all_preds,   axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    pred_flat   = all_preds.reshape(-1)
    target_flat = all_targets.reshape(-1)

    dice_pc, mean_dice = dice_coefficient(pred_flat, target_flat)
    iou_pc,  miou      = mean_iou(pred_flat, target_flat)

    biou_per_class_list = [[] for _ in range(NUM_CLASSES)]
    for i in range(min(10, len(all_preds))):
        biou_pc, _ = boundary_iou_all_classes(all_preds[i], all_targets[i])
        for c in range(NUM_CLASSES):
            biou_per_class_list[c].append(biou_pc[c])
    biou_pc   = [float(np.mean(v)) for v in biou_per_class_list]
    mean_biou = float(np.mean(biou_pc))

    print(f'\n── {split_name} ─────────────────────────────────────────────')
    print(f'{"Class":20s}  {"Dice":>8s}  {"IoU":>8s}  {"BIoU":>8s}')
    print('-'*52)
    for c, name in enumerate(CLASS_NAMES):
        print(f'{name:20s}  {dice_pc[c]:>8.4f}  '
              f'{iou_pc[c]:>8.4f}  {biou_pc[c]:>8.4f}')
    print('-'*52)
    print(f'{"Mean":20s}  {mean_dice:>8.4f}  {miou:>8.4f}  {mean_biou:>8.4f}')

    return {
        'dice_per_class': dice_pc, 'mean_dice': mean_dice,
        'iou_per_class' : iou_pc,  'miou'     : miou,
        'biou_per_class': biou_pc, 'mean_biou': mean_biou,
        'all_preds'     : all_preds,
        'all_targets'   : all_targets
    }


test_results = evaluate_on_tiles(
    model, test_loader, device,
    'Test Tiles (Training Reach 2024)'
)

test_metrics = {k: v for k, v in test_results.items()
                if k not in ['all_preds','all_targets']}
with open(os.path.join(OUTPUTS_DIR,'segformer_test_metrics.json'),'w') as f:
    json.dump(test_metrics, f, indent=2)
print('Test metrics saved.')

In [ ]:
# ── Visualise test tile predictions ──────────────────────────────────────
def visualise_predictions(all_preds, all_targets, test_dataset,
                           n_samples=6, save_path=None):
    cmap = mcolors.ListedColormap(CLASS_COLORS)
    norm = mcolors.BoundaryNorm([0,1,2,3,4], cmap.N)
    indices = random.sample(range(len(all_preds)), min(n_samples, len(all_preds)))

    fig, axes = plt.subplots(n_samples, 4, figsize=(20, n_samples*4))
    if n_samples == 1:
        axes = axes[np.newaxis, :]

    for col, title in enumerate(['True Colour','NIR False Colour',
                                  'Ground Truth','Prediction']):
        axes[0, col].set_title(title, fontsize=10, fontweight='bold')

    for row, idx in enumerate(indices):
        tile_id  = test_dataset.tile_ids[idx]
        image_np = np.load(
            os.path.join(get_tile_dir(tile_id), 'image.npy')
        ).astype(np.float32)

        rgb = image_np[:,:,[2,1,0]]
        rgb = np.clip((rgb-200)/(2500-200), 0, 1)
        axes[row,0].imshow(rgb)
        axes[row,0].set_ylabel(f'Tile {tile_id[-4:]}', fontsize=8)
        axes[row,0].axis('off')

        nir = image_np[:,:,[3,2,1]]
        nir = np.clip((nir-200)/(3500-200), 0, 1)
        axes[row,1].imshow(nir)
        axes[row,1].axis('off')

        axes[row,2].imshow(all_targets[idx], cmap=cmap, norm=norm)
        axes[row,2].axis('off')

        im = axes[row,3].imshow(all_preds[idx], cmap=cmap, norm=norm)
        axes[row,3].axis('off')

        _, tile_dice = dice_coefficient(all_preds[idx], all_targets[idx])
        axes[row,3].set_xlabel(f'Dice={tile_dice:.3f}', fontsize=8)

    cbar = fig.colorbar(im, ax=axes[:,3], ticks=[0.5,1.5,2.5,3.5],
                        fraction=0.02, pad=0.04)
    cbar.ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    plt.suptitle('SegFormer-B2 — Test Tile Predictions (Training Reach 2024)',
                 fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved → {save_path}')
    plt.show()


visualise_predictions(
    test_results['all_preds'],
    test_results['all_targets'],
    test_dataset,
    n_samples=6,
    save_path=os.path.join(FIGURES_DIR,'segformer_test_predictions.png')
)

## 12. Inference Reach Prediction (2025)
Streaming inference — processes scene row by row to avoid RAM crash.
Braidplain constraint applied at logit level tile by tile.
No accuracy metrics — evaluation via morphological integrity cross-sections.

In [ ]:
def evaluate_domain_shift(model, inference_dir, norm_mean, norm_std,
                           year='2025', device='cuda'):
    """
    Streaming full-scene inference for SegFormer.
    Identical pipeline to Script 2A Attention U-Net —
    ensures fair comparison for morphological integrity assessment.

    Key differences from Attention U-Net:
    - Uses segformer_forward() with bilinear upsampling
    - Saves as segformer_pred_{year}_braidplain.tif
    """
    scene_path = os.path.join(inference_dir, f'inference_{year}_9band_merged.tif')
    if not os.path.exists(scene_path):
        scene_path = os.path.join(inference_dir, f'inference_{year}_9band.tif')

    print(f'Loading: {os.path.basename(scene_path)}')

    src    = rasterio.open(scene_path)
    H, W   = src.height, src.width
    print(f'Scene size: {H} x {W}')

    # ── Load braidplain mask ──────────────────────────────────────────
    val_candidates = sorted(glob.glob(
        os.path.join(inference_dir, f'inference_{year}_validation_mask*.tif')
    ))
    if not val_candidates:
        print('ERROR: No validation mask found — needed for braidplain constraint')
        src.close()
        return None

    with rasterio.open(val_candidates[0]) as msrc:
        braidplain = msrc.read(1) > 0
    print(f'Braidplain pixels: {braidplain.sum():,}')

    # ── Scene-specific normalisation ──────────────────────────────────
    # Load full scene to compute stats then discard
    print('Computing scene-specific normalisation...')
    image_full = src.read().astype(np.float32)  # (9, H, W)

    has_nan  = np.isnan(image_full[0]).any()
    has_zero = (image_full[0] == 0).any()
    if has_nan:
        valid_mask = ~np.isnan(image_full[0])
    elif has_zero:
        valid_mask = image_full[0] != 0
    else:
        valid_mask = np.ones((H, W), dtype=bool)

    scene_mean = np.zeros(IN_CHANNELS, dtype=np.float32)
    scene_std  = np.zeros(IN_CHANNELS, dtype=np.float32)

    print(f'{"Band":6s}  {"Train Mean":>12s}  {"Scene Mean":>12s}')
    print('-'*36)
    for c in range(IN_CHANNELS):
        band  = image_full[c].astype(np.float64)
        band  = np.where(np.isnan(band), 0, band)
        valid = band[valid_mask]
        if len(valid) > 0 and not np.all(valid == 0):
            scene_mean[c] = float(np.mean(valid))
            scene_std[c]  = float(np.std(valid))
            if scene_std[c] < 1e-6:
                scene_std[c] = 1.0
        else:
            scene_mean[c] = norm_mean[c]
            scene_std[c]  = norm_std[c]
        print(f'{BAND_NAMES[c]:6s}  {norm_mean[c]:>12.2f}  {scene_mean[c]:>12.2f}')

    del image_full
    gc.collect()

    # ── Output raster setup ───────────────────────────────────────────
    profile = src.profile.copy()
    profile.update(dtype=rasterio.uint8, count=1, compress='lzw', nodata=0)

    out_path = os.path.join(OUTPUTS_DIR, f'segformer_pred_{year}_braidplain.tif')
    dst = rasterio.open(out_path, 'w', **profile)

    # ── Streaming sliding window inference ───────────────────────────
    tile_size = TILE_SIZE
    overlap   = 64
    step      = tile_size - overlap

    # Accumulation arrays — float16 to save RAM
    logit_sum = np.zeros((NUM_CLASSES, H, W), dtype=np.float16)
    count_map = np.zeros((H, W),              dtype=np.uint8)

    model.eval()
    n_tiles_h   = (H + step - 1) // step
    n_tiles_w   = (W + step - 1) // step
    total_tiles = n_tiles_h * n_tiles_w
    processed   = 0

    print('\nRunning streaming inference...')

    with torch.no_grad():
        y = 0
        while y < H:
            y_end   = min(y + tile_size, H)
            y_start = max(0, y_end - tile_size)
            x = 0
            while x < W:
                x_end   = min(x + tile_size, W)
                x_start = max(0, x_end - tile_size)

                # Read tile using rasterio window
                window = rasterio.windows.Window(
                    x_start, y_start,
                    x_end - x_start,
                    y_end - y_start
                )
                patch = src.read(window=window).astype(np.float32)
                ph, pw = patch.shape[1], patch.shape[2]

                # Scene-specific normalisation per tile
                for c in range(IN_CHANNELS):
                    band = patch[c]
                    band = np.where(np.isnan(band), scene_mean[c], band)
                    band = np.where(band == 0,      scene_mean[c], band)
                    patch[c] = (band - scene_mean[c]) / scene_std[c]

                # Pad to tile_size if needed
                if ph < tile_size or pw < tile_size:
                    padded = np.zeros(
                        (IN_CHANNELS, tile_size, tile_size),
                        dtype=np.float32
                    )
                    padded[:, :ph, :pw] = patch
                    patch = padded

                tensor = torch.from_numpy(patch).unsqueeze(0).to(device)

                # SegFormer forward with upsampling
                out    = model(pixel_values=tensor)
                logits = F.interpolate(
                    out.logits,
                    size=(tile_size, tile_size),
                    mode='bilinear',
                    align_corners=False
                )
                logits_np = logits.squeeze(0).cpu().numpy().astype(np.float16)

                # Accumulate
                logit_sum[:, y_start:y_end, x_start:x_end] += \
                    logits_np[:, :y_end-y_start, :x_end-x_start]
                count_map[y_start:y_end, x_start:x_end] += 1

                del tensor, out, logits, logits_np
                x += step

            processed += n_tiles_w
            if processed % (n_tiles_w * 10) == 0:
                print(f'  {100*processed/total_tiles:.0f}% '
                      f'({processed}/{total_tiles} tiles)')
            y += step

    src.close()

    # ── Average and argmax ────────────────────────────────────────────
    count_map = np.maximum(count_map.astype(np.float16), 1)
    logit_avg = logit_sum / count_map
    del logit_sum, count_map
    gc.collect()

    pred_mask = np.argmax(logit_avg, axis=0).astype(np.uint8)

    # ── Braidplain constraint ─────────────────────────────────────────
    # Suppress background inside braidplain — geomorphologically justified
    print('Applying braidplain constraint...')
    bp_bg       = braidplain & (pred_mask == 0)
    n_bg_before = int(bp_bg.sum())
    print(f'Background inside braidplain before: {n_bg_before:,}')

    if n_bg_before > 0:
        logit_c = logit_avg.copy()
        logit_c[0][braidplain] = -1e4
        new_pred = np.argmax(logit_c, axis=0).astype(np.uint8)
        pred_mask[bp_bg] = new_pred[bp_bg]
        del logit_c, new_pred
        gc.collect()
        print(f'Background inside braidplain after : '
              f'{int((braidplain & (pred_mask==0)).sum()):,}')

    del logit_avg
    gc.collect()

    # ── Clip to braidplain ────────────────────────────────────────────
    pred_mask[~braidplain] = 0

    # ── Save ─────────────────────────────────────────────────────────
    dst.write(pred_mask.astype(np.uint8), 1)
    dst.close()
    print(f'Saved → {out_path}')

    # ── Class distribution inside braidplain ──────────────────────────
    pred_bp = pred_mask[braidplain]
    total   = braidplain.sum()
    print('\n── Class Distribution (Braidplain) ─────────────────────────')
    for c, name in enumerate(CLASS_NAMES):
        n   = int(np.sum(pred_bp == c))
        pct = 100 * n / total
        print(f'  Class {c} {name:16s}: {n:>10,} px  ({pct:.1f}%)')

    del pred_mask, pred_bp
    gc.collect()

    # ── Downsampled visualisation ─────────────────────────────────────
    print('\nGenerating visualisation...')
    with rasterio.open(out_path) as s:
        pred_viz = s.read(1).astype(np.int64)
    with rasterio.open(val_candidates[0]) as s:
        val_viz  = s.read(1).astype(np.int64)

    if pred_viz.shape != val_viz.shape:
        min_h = min(pred_viz.shape[0], val_viz.shape[0])
        min_w = min(pred_viz.shape[1], val_viz.shape[1])
        pred_viz = pred_viz[:min_h, :min_w]
        val_viz  = val_viz[:min_h,  :min_w]

    scale  = min(2000 / max(H, W), 1.0)
    step_h = max(1, int(1 / scale))
    step_w = max(1, int(1 / scale))
    pred_ds = pred_viz[::step_h, ::step_w]
    val_ds  = val_viz[::step_h,  ::step_w]
    del pred_viz, val_viz
    gc.collect()

    cmap   = mcolors.ListedColormap(CLASS_COLORS)
    norm_c = mcolors.BoundaryNorm([0,1,2,3,4], cmap.N)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(
        f'SegFormer-B2 — Inference Reach Prediction\n'
        f'Tezpur–Guwahati {year} | Braidplain Clipped',
        fontsize=12, fontweight='bold'
    )
    axes[0].imshow(val_ds,  cmap=cmap, norm=norm_c, interpolation='nearest')
    axes[0].set_title('Validation Label (Index-Based Weak Label)')
    axes[0].axis('off')
    im = axes[1].imshow(pred_ds, cmap=cmap, norm=norm_c, interpolation='nearest')
    axes[1].set_title('SegFormer-B2 Prediction')
    axes[1].axis('off')
    cbar = fig.colorbar(im, ax=axes, ticks=[0.5,1.5,2.5,3.5],
                        fraction=0.02, pad=0.02)
    cbar.ax.set_yticklabels(CLASS_NAMES, fontsize=9)
    fig_path = os.path.join(FIGURES_DIR, f'segformer_pred_{year}_braidplain.png')
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Figure saved → {fig_path}')

    del pred_ds, val_ds
    gc.collect()

    # ── Save scene normalisation stats for Script 3 ───────────────────
    stats_path = os.path.join(OUTPUTS_DIR, f'scene_norm_stats_{year}.json')
    with open(stats_path, 'w') as f:
        json.dump({
            'year'      : year,
            'scene_mean': scene_mean.tolist(),
            'scene_std' : scene_std.tolist()
        }, f, indent=2)
    print(f'Scene norm stats saved → {stats_path}')
    print('\nInference complete. Evaluate via morphological integrity cross-sections.')

    return out_path


pred_path = evaluate_domain_shift(
    model, INFERENCE_DIR, NORM_MEAN, NORM_STD,
    year='2025', device=device
)

## 13. Post-Inference Visualisation Cell

In [ ]:
# ── Run this cell separately if inference cell crashed before visualisation
# Loads from saved GeoTIFFs — no large arrays needed in memory

def visualise_from_tiff(year='2025'):
    pred_path = os.path.join(OUTPUTS_DIR, f'segformer_pred_{year}_braidplain.tif')
    val_candidates = sorted(glob.glob(
        os.path.join(INFERENCE_DIR, f'inference_{year}_validation_mask*.tif')
    ))

    if not os.path.exists(pred_path):
        print(f'ERROR: {pred_path} not found. Run evaluate_domain_shift() first.')
        return

    with rasterio.open(pred_path) as s:
        pred = s.read(1).astype(np.int64)
        H, W = s.height, s.width
    with rasterio.open(val_candidates[0]) as s:
        val = s.read(1).astype(np.int64)

    if pred.shape != val.shape:
        min_h = min(pred.shape[0], val.shape[0])
        min_w = min(pred.shape[1], val.shape[1])
        pred = pred[:min_h, :min_w]
        val  = val[:min_h,  :min_w]

    scale  = min(2000 / max(H, W), 1.0)
    step_h = max(1, int(1 / scale))
    step_w = max(1, int(1 / scale))

    cmap   = mcolors.ListedColormap(CLASS_COLORS)
    norm_c = mcolors.BoundaryNorm([0,1,2,3,4], cmap.N)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(
        f'SegFormer-B2 — Inference Reach Prediction\n'
        f'Tezpur–Guwahati {year} | Braidplain Clipped',
        fontsize=12, fontweight='bold'
    )
    axes[0].imshow(val[::step_h,  ::step_w], cmap=cmap, norm=norm_c,
                   interpolation='nearest')
    axes[0].set_title('Validation Label')
    axes[0].axis('off')
    im = axes[1].imshow(pred[::step_h, ::step_w], cmap=cmap, norm=norm_c,
                        interpolation='nearest')
    axes[1].set_title('SegFormer-B2 Prediction')
    axes[1].axis('off')
    cbar = fig.colorbar(im, ax=axes, ticks=[0.5,1.5,2.5,3.5],
                        fraction=0.02, pad=0.02)
    cbar.ax.set_yticklabels(CLASS_NAMES, fontsize=9)
    fig.savefig(
        os.path.join(FIGURES_DIR, f'segformer_pred_{year}_braidplain.png'),
        dpi=150, bbox_inches='tight'
    )
    plt.show()
    plt.close(fig)
    del pred, val
    gc.collect()
    print('Done.')


visualise_from_tiff(year='2025')

## 14. Model Comparison Summary
Compare SegFormer vs Attention U-Net test tile metrics side by side.
Full model selection based on morphological integrity cross-section assessment.

In [ ]:
# ── Load Attention U-Net test metrics for comparison ─────────────────────
attn_metrics_path = os.path.join(ATTN_OUTPUTS, 'attention_unet_test_metrics.json')

if os.path.exists(attn_metrics_path):
    with open(attn_metrics_path) as f:
        attn_metrics = json.load(f)

    print('='*70)
    print('MODEL COMPARISON — TEST TILES (Training Reach 2024)')
    print('='*70)
    print(f'{"Class":20s}  {"Attn Dice":>10s}  {"Seg Dice":>10s}  '
          f'{"Attn IoU":>10s}  {"Seg IoU":>10s}')
    print('-'*66)
    for c, name in enumerate(CLASS_NAMES):
        ad = attn_metrics['dice_per_class'][c]
        sd = test_metrics['dice_per_class'][c]
        ai = attn_metrics['iou_per_class'][c]
        si = test_metrics['iou_per_class'][c]
        better_d = '✓' if sd >= ad else ' '
        print(f'{name:20s}  {ad:>10.4f}  {sd:>10.4f}{better_d}  '
              f'{ai:>10.4f}  {si:>10.4f}')
    print('-'*66)
    print(f'{"Mean":20s}  '
          f'{attn_metrics["mean_dice"]:>10.4f}  '
          f'{test_metrics["mean_dice"]:>10.4f}  '
          f'{attn_metrics["miou"]:>10.4f}  '
          f'{test_metrics["miou"]:>10.4f}')
    print()
    print('NOTE: Final model selection based on morphological integrity')
    print('      cross-section regression (R²) on inference reach 2025.')
    print('      Higher R² for braidplain width, channel width, bar width,')
    print('      and channel thread count → selected for Script 3.')
    print('='*70)
else:
    print('Attention U-Net metrics not found.')
    print('Run Script 2A test evaluation first.')
    print()
    print('SegFormer-B2 test tile results:')
    print(f'  Mean Dice : {test_metrics["mean_dice"]:.4f}')
    print(f'  Mean IoU  : {test_metrics["miou"]:.4f}')
    print(f'  Mean BIoU : {test_metrics["mean_biou"]:.4f}')